### Exercise for building a trigram model

In [1]:
!wget https://raw.githubusercontent.com/karpathy/makemore/refs/heads/master/names.txt

--2026-03-06 10:31:03--  https://raw.githubusercontent.com/karpathy/makemore/refs/heads/master/names.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.109.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 228145 (223K) [text/plain]
Saving to: ‘names.txt’

names.txt           100%[===================>] 222.80K  --.-KB/s    in 0.02s   

2026-03-06 10:31:03 (12.3 MB/s) - ‘names.txt’ saved [228145/228145]



In [2]:
with open("names.txt", "r") as f:
  words = [line.strip() for line in f]

words[:10]

['emma',
 'olivia',
 'ava',
 'isabella',
 'sophia',
 'charlotte',
 'mia',
 'amelia',
 'harper',
 'evelyn']

In [3]:
import torch

In [4]:
P = torch.ones(27,27,27, dtype=torch.float64)
P.shape

torch.Size([27, 27, 27])

In [5]:
def char_to_idx(c):
  return 0 if c == "." else (ord(c) - ord('a') + 1)

def idx_to_char(idx):
  return '.' if idx == 0 else chr(ord('a') + idx - 1)

In [ ]:
for word in words:
  word = f".{word}."
  for char1, char2, char3 in zip(word, word[1:], word[2:]):
    char1_idx = char_to_idx(char1)
    char2_idx = char_to_idx(char2)
    char3_idx = char_to_idx(char3)

    P[char1_idx, char2_idx, char3_idx] += 1

In [ ]:
P

tensor([[[  1.,   1.,   1.,  ...,   1.,   1.,   1.],
         [  1., 208., 191.,  ...,  28., 174., 153.],
         [  1., 170.,   1.,  ...,   1.,   5.,   1.],
         ...,
         [  1.,  58.,   1.,  ...,   2.,  18.,  12.],
         [  1., 247.,   1.,  ...,   1.,   1.,   3.],
         [  1., 457.,   1.,  ...,   1.,  92.,   2.]],

        [[  1.,   1.,   1.,  ...,   1.,   1.,   1.],
         [ 41.,   1.,   6.,  ...,   1.,  21.,  12.],
         [ 37.,  29.,  21.,  ...,   1.,  13.,   1.],
         ...,
         [ 12.,   6.,   1.,  ...,  18.,   7.,   4.],
         [164., 390.,  14.,  ...,   1.,  17.,  41.],
         [ 39., 124.,   1.,  ...,   1.,  13.,  23.]],

        [[  1.,   1.,   1.,  ...,   1.,   1.,   1.],
         [ 47.,   6.,   6.,  ...,   5.,  32.,   5.],
         [  2.,   9.,   1.,  ...,   1.,  10.,   1.],
         ...,
         [  1.,   1.,   1.,  ...,   1.,   1.,   1.],
         [ 56.,   5.,   2.,  ...,   1.,   1.,   1.],
         [  1.,   1.,   1.,  ...,   1.,   1.,   1.]],

Computing probabilities instead

In [ ]:
P /= P.sum(dim=2, keepdim=True)

In [ ]:
P[0,0].sum()

tensor(1., dtype=torch.float64)

In [ ]:
P[0,:].sum()

tensor(27., dtype=torch.float64)

In [ ]:
torch.set_printoptions(sci_mode=False)
P

tensor([[[0.0370, 0.0370, 0.0370,  ..., 0.0370, 0.0370, 0.0370],
         [0.0002, 0.0469, 0.0430,  ..., 0.0063, 0.0392, 0.0345],
         [0.0008, 0.1275, 0.0008,  ..., 0.0008, 0.0038, 0.0008],
         ...,
         [0.0062, 0.3602, 0.0062,  ..., 0.0124, 0.1118, 0.0745],
         [0.0018, 0.4395, 0.0018,  ..., 0.0018, 0.0018, 0.0053],
         [0.0010, 0.4780, 0.0010,  ..., 0.0010, 0.0962, 0.0021]],

        [[0.0370, 0.0370, 0.0370,  ..., 0.0370, 0.0370, 0.0370],
         [0.0703, 0.0017, 0.0103,  ..., 0.0017, 0.0360, 0.0206],
         [0.0651, 0.0511, 0.0370,  ..., 0.0018, 0.0229, 0.0018],
         ...,
         [0.0574, 0.0287, 0.0048,  ..., 0.0861, 0.0335, 0.0191],
         [0.0790, 0.1878, 0.0067,  ..., 0.0005, 0.0082, 0.0197],
         [0.0844, 0.2684, 0.0022,  ..., 0.0022, 0.0281, 0.0498]],

        [[0.0370, 0.0370, 0.0370,  ..., 0.0370, 0.0370, 0.0370],
         [0.1351, 0.0172, 0.0172,  ..., 0.0144, 0.0920, 0.0144],
         [0.0308, 0.1385, 0.0154,  ..., 0.0154, 0.1538, 0.

Sample words from multinomial distribution

In [ ]:
g = torch.Generator().manual_seed(2147483647)
x = torch.randint(1, 27, (1,)).item()
idx = torch.multinomial(P[0,x], num_samples=1, replacement=True, generator=g)
idx, idx.shape

(tensor([18]), torch.Size([1]))

In [ ]:
x = torch.randint(1, 27, (1,)).item()
x

In [24]:
g = torch.Generator().manual_seed(2147483647)

gen_num_words = 5

for _ in range(gen_num_words):
  idx1 = 0
  idx2 = torch.randint(1, 27, (1,)).item()

  out = []
  out.append(idx_to_char(idx2))

  while True:
    idx3 = torch.multinomial(P[idx1,idx2], num_samples=1, replacement=True, generator=g).item()

    if idx3 == 0:
      break
    out.append(idx_to_char(idx3))
    idx1, idx2 = idx2, idx3
  print("".join(out))

ucexzm
ozoglkurkicqzktyhwmvmzimjttainrlkfukzkktda
ksfcxvpubjtbhrmgotzx
diczixqctvujkwptedogkkjemkmmsidguenkbvgynywftbspmhwcivgbvtahlvsu
mdsdxxblnwglhpyiw


## Calculating loss

In [ ]:

n = 0
nll = 0

for word in words:
  word = f".{word}."
  for char1, char2, char3 in zip(word, word[1:], word[2:]):
    char1_idx = char_to_idx(char1)
    char2_idx = char_to_idx(char2)
    char3_idx = char_to_idx(char3)

    nll += torch.log(P[char1_idx, char2_idx, char3_idx]) # This means that for char1, char2 we consider char3 as prediction
    n += 1 # Counting each trigram

nll *= -1
print(f"{nll=}")

print(f"Avg nll {nll.item()/n}") # This shows avg NLL considering freq matrix approach as one of the models

nll=tensor(410480.1701, dtype=torch.float64)
Avg nll 2.093079857518903


Training a nn model using gradient descent

In [ ]:
## How to define inputs and W and of what dimensions:
test = torch.tensor([[1,2]])
test.shape

torch.Size([1, 2])

In [6]:
import torch.nn.functional as F

In [ ]:
oh = F.one_hot(test, num_classes=27)
oh.shape

torch.Size([1, 2, 27])

In [7]:
## Preparing dataset
X, Y = [], []
for word in words:
  word = f".{word}."
  for char1, char2, char3 in zip(word, word[1:], word[2:]):
    X.append([char_to_idx(char1), char_to_idx(char2)])
    Y.append(char_to_idx(char3))

X = torch.tensor(X)
Y = torch.tensor(Y)
X.shape, Y.shape

(torch.Size([196113, 2]), torch.Size([196113]))

In [ ]:
xenc = F.one_hot(X, num_classes=27).double()
xenc[0], xenc.shape, xenc.dtype

(tensor([[1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0., 0., 0.]], dtype=torch.float64),
 torch.Size([196113, 2, 27]),
 torch.float64)

In [ ]:
# Reshape
xenc = xenc.reshape(xenc.shape[0], -1)
xenc[0], xenc.shape

(tensor([1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        dtype=torch.float64),
 torch.Size([196113, 54]))

In [ ]:
yenc = F.one_hot(Y, num_classes=27).double()
yenc.shape, yenc.dtype

(torch.Size([196113, 27]), torch.float64)

In [ ]:
W = torch.randn(54,27, dtype=torch.float64, requires_grad=True)
W, W.shape

(tensor([[-0.8916,  0.1418, -1.5649,  ...,  0.7823, -0.4109, -1.7113],
         [ 2.1375, -0.0577,  0.7956,  ...,  0.8384,  0.8524, -0.1485],
         [ 2.2395, -0.8816,  0.7390,  ...,  0.2530, -2.5995, -0.8528],
         ...,
         [-0.1059,  0.8574, -0.7059,  ...,  0.2452, -0.1293,  0.9433],
         [ 0.3374, -0.4493,  0.2637,  ...,  0.0592,  0.5294,  0.6035],
         [ 0.4254,  1.4050,  1.1248,  ...,  1.0036,  0.1462,  0.5286]],
        dtype=torch.float64, requires_grad=True),
 torch.Size([54, 27]))

In [ ]:
## Creating W
step = 50
runs = 100

for run in range(runs):
  # Forward pass
  logits = xenc @ W #[nin, 27]
  #probs = torch.nn.Softmax(logits, dim=1)
  log_probs = F.log_softmax(logits, dim=1)

  loss = -(yenc * log_probs).sum(dim=1).mean() + 0.1*(W**2).mean()

  W.grad = None
  loss.backward()

  W.data -= step * W.grad

  print(f"Run: {run+1} Loss: {loss}")



Run: 1 Loss: 4.327683357627846
Run: 2 Loss: 3.524967266247848
Run: 3 Loss: 3.166374693959162
Run: 4 Loss: 2.971608353795685
Run: 5 Loss: 2.8483139682849186
Run: 6 Loss: 2.7614863148363162
Run: 7 Loss: 2.6975171259444517
Run: 8 Loss: 2.648768122498095
Run: 9 Loss: 2.610476953526762
Run: 10 Loss: 2.579595667432384
Run: 11 Loss: 2.554156527255993
Run: 12 Loss: 2.5328207229119246
Run: 13 Loss: 2.514655797659015
Run: 14 Loss: 2.4989894999716302
Run: 15 Loss: 2.4853317683602225
Run: 16 Loss: 2.4733150950252156
Run: 17 Loss: 2.4626606673242226
Run: 18 Loss: 2.4531509756503054
Run: 19 Loss: 2.4446145262483117
Run: 20 Loss: 2.4369128186831324
Run: 21 Loss: 2.4299332546662633
Run: 22 Loss: 2.4235825532162902
Run: 23 Loss: 2.417783183466443
Run: 24 Loss: 2.412469722600446
Run: 25 Loss: 2.4075868705900647
Run: 26 Loss: 2.4030872752054595
Run: 27 Loss: 2.398930340747092
Run: 28 Loss: 2.3950808649638744
Run: 29 Loss: 2.391508292002367
Run: 30 Loss: 2.38818583311674
Run: 31 Loss: 2.3850899847745892
R

In [ ]:
# Do a generation as well
# Starting with .
for _ in range(5):
  idx1 = 0
  idx2 = torch.randint(1, 27, (1,)).item()
  out = []
  char1 = idx_to_char(idx2)
  #print(char1)
  out.append(char1)
  while True:
    # Forward pass
    xs_enc = F.one_hot(torch.tensor([[idx1,idx2]]), num_classes=27).double() #[2,27]
    xs_enc = xs_enc.reshape(1,-1) # [1,54]
    logits = xs_enc @ W #[1, 54] * [54, 27] = [1, 27]
    probs = F.softmax(logits, dim=1)
    #print(probs)
    idx3 = torch.multinomial(probs, num_samples=1, replacement=True, generator=g)
    c = idx_to_char(idx3)
    #print(c)
    if c == ".":
      break
    else:
      out.append(c)
      idx1, idx2 = idx2, idx3
  print("".join(out))
  #print("New word\n")


dakavirny
jatlspih
monden
ftahlasu
wasdrx


#### Doing [27*27] instead of [27+27]

In [8]:
26 * 26

676

In [9]:
## Preparing dataset
X, Y = [], []
num_classes = 2
for word in words:
  word = f".{word}."
  for char1, char2, char3 in zip(word, word[1:], word[2:]):
    combined_idx = char_to_idx(char1)*num_classes + char_to_idx(char2)
    X.append(combined_idx)
    Y.append(char_to_idx(char3))

X = torch.tensor(X)
Y = torch.tensor(Y)
X.shape, Y.shape

(torch.Size([196113]), torch.Size([196113]))

In [10]:
xenc = F.one_hot(X, num_classes=27*27).double()
yenc = F.one_hot(Y, num_classes=27).double()
xenc.shape, xenc.dtype, yenc.shape, yenc.dtype

(torch.Size([196113, 729]),
 torch.float64,
 torch.Size([196113, 27]),
 torch.float64)

In [11]:
W = torch.randn(729,27, dtype=torch.float64, requires_grad=True)
W.dtype, W.shape

(torch.float64, torch.Size([729, 27]))

In [26]:
## Creating W
step = 200
runs = 100

for run in range(runs):
  # Forward pass
  logits = xenc @ W #[nin, 27]
  #probs = torch.nn.Softmax(logits, dim=1)
  log_probs = F.log_softmax(logits, dim=1)

  loss = -(yenc * log_probs).sum(dim=1).mean() + 0.001*(W**2).mean()

  W.grad = None
  loss.backward()

  W.data -= step * W.grad

  print(f"Run: {run+1} Loss: {loss}")


Run: 1 Loss: 2.4298678942669096
Run: 2 Loss: 2.429800860068022
Run: 3 Loss: 2.4297349164229787
Run: 4 Loss: 2.4296700361348154
Run: 5 Loss: 2.429606193024112
Run: 6 Loss: 2.429543361823466
Run: 7 Loss: 2.42948151812503
Run: 8 Loss: 2.4294206383387906
Run: 9 Loss: 2.4293606996555273
Run: 10 Loss: 2.4293016800128253
Run: 11 Loss: 2.429243558063284
Run: 12 Loss: 2.429186313144623
Run: 13 Loss: 2.429129925251405
Run: 14 Loss: 2.429074375008257
Run: 15 Loss: 2.429019643644434
Run: 16 Loss: 2.4289657129696614
Run: 17 Loss: 2.4289125653511463
Run: 18 Loss: 2.4288601836917003
Run: 19 Loss: 2.4288085514088977
Run: 20 Loss: 2.428757652415219
Run: 21 Loss: 2.4287074710991132
Run: 22 Loss: 2.428657992306939
Run: 23 Loss: 2.428609201325727
Run: 24 Loss: 2.4285610838667258
Run: 25 Loss: 2.4285136260496896
Run: 26 Loss: 2.428466814387864
Run: 27 Loss: 2.4284206357736395
Run: 28 Loss: 2.428375077464836
Run: 29 Loss: 2.4283301270715816
Run: 30 Loss: 2.428285772543766
Run: 31 Loss: 2.428242002159032
Run

In [27]:
# Do a generation as well
# Starting with .
g = torch.Generator().manual_seed(2147483647)

for _ in range(5):
  idx1 = 0
  idx2 = 1
  out = []
  char1 = idx_to_char(idx2)
  out.append(char1)
  while True:
    # Forward pass
    combined_idx = idx1*27 + idx2
    xs_enc = F.one_hot(torch.tensor([combined_idx]), num_classes=27*27).double() #[1,729]
    logits = xs_enc @ W #[1, 729] * [729, 27] = [1, 27]
    probs = F.softmax(logits, dim=1)
    idx3 = torch.multinomial(probs, num_samples=1, replacement=True, generator=g)
    c = idx_to_char(idx3)

    if c == ".":
      break
    else:
      out.append(c)
      idx1, idx2 = idx2, idx3
  print("".join(out))


adexzm
alegpsurkroczvtyhkmv
alirjtnwinrtkzdkzkhgam
arncxvpuxjtxhrmgotzi
amozumq


#### E04
we saw that our 1-hot vectors merely select a row of W, so producing these vectors explicitly feels wasteful. Can you delete our use of F.one_hot in favor of simply indexing into rows of W?

In [63]:
# Going back to a bigram model
## Preparing dataset


X, Y = [], []
for word in words:
  word = f".{word}."
  for char1, char2 in zip(word, word[1:]):
    X.append(char_to_idx(char1))
    Y.append(char_to_idx(char2))

X = torch.tensor(X)
Y = torch.tensor(Y)
X.shape, Y.shape

(torch.Size([228146]), torch.Size([228146]))

In [33]:
# One hot encoding
Y_enc = F.one_hot(Y, num_classes=27).double()


(torch.Size([228146, 27]), torch.Size([228146, 27]))

In [64]:
W = torch.randn(27, 27, requires_grad=True, dtype=torch.float64)
W.shape

torch.Size([27, 27])

In [54]:
TW = torch.randn(3,3, dtype=torch.float64)
TX = torch.randint(0,3, (5,), dtype=torch.int64)
TW, TX

(tensor([[-0.3074,  0.2227,  1.0017],
         [-0.4862,  1.8414,  0.8408],
         [-1.6441, -1.7091, -3.0148]], dtype=torch.float64),
 tensor([2, 2, 1, 2, 2]))

In [56]:
TW[TX]

tensor([[-1.6441, -1.7091, -3.0148],
        [-1.6441, -1.7091, -3.0148],
        [-0.4862,  1.8414,  0.8408],
        [-1.6441, -1.7091, -3.0148],
        [-1.6441, -1.7091, -3.0148]], dtype=torch.float64)

In [61]:
runs = 100
step = 100

for _ in range(runs):

  # Forward pass
  logits = W[X] #### This is the ANSWER (directly picks up the row corresponding to index)
  counts = logits.exp()
  probs = counts / counts.sum(dim=1, keepdim=True)
  loss = -probs[torch.arange(X.shape[0]), Y].log().mean() + 0.001 * (W**2).mean()
  print(f"Loss: {loss.item()}")

  # Backward pass
  W.grad = None
  loss.backward()

  # Grad
  W.data -= step * W.grad

Loss: 2.6100487500709235
Loss: 2.57845119362309
Loss: 2.575604447183751
Loss: 2.5674907232420234
Loss: 2.5810997687806387
Loss: 2.552918703411062
Loss: 2.5533168764523246
Loss: 2.5477148859917453
Loss: 2.563867776371961
Loss: 2.537189870517343
Loss: 2.5391285693629553
Loss: 2.5349945846036643
Loss: 2.552568039152525
Loss: 2.526666746829102
Loss: 2.5294400371944294
Loss: 2.5262500701312347
Loss: 2.544713712578754
Loss: 2.5192321683111865
Loss: 2.5224872383203767
Loss: 2.519919246759924
Loss: 2.5389698628438118
Loss: 2.5137245223443294
Loss: 2.517272680486446
Loss: 2.5151241947906096
Loss: 2.5345768328476677
Loss: 2.509476390000301
Loss: 2.5132166720828755
Loss: 2.511359946988602
Loss: 2.531098364395578
Loss: 2.5060973241414675
Loss: 2.5099745533537745
Loss: 2.5083271833811183
Loss: 2.5282756581283117
Loss: 2.5033495243107855
Loss: 2.5073313415903162
Loss: 2.5058383744507764
Loss: 2.525945506381219
Loss: 2.5010790598121977
Loss: 2.5051443534403406
Loss: 2.5037678471993026
Loss: 2.5239973

##### E05
look up and use F.cross_entropy instead. You should achieve the same result. Can you think of why we'd prefer to use F.cross_entropy instead?

In [ ]:
### Alternative ways to calculate loss

# Negative log likelihood
# Cross entropy

In [65]:
# Manual way of calculating
W = torch.randn(27, 27, requires_grad=True, dtype=torch.float64)

runs = 100
step = 100
N = X.shape[0]

for _ in range(runs):

  # Forward pass
  logits = W[X]
  counts = logits.exp()
  probs = counts / counts.sum(dim=1, keepdim=True) # <---------- SOFTMAX
  log_probs = probs.log()
  loss = -log_probs[torch.arange(N), Y].mean() + 0.001 * (W**2).mean()  # <---------- DIRECT INDEXING
  print(f"Loss: {loss.item()}")

  # Backward pass
  W.grad = None
  loss.backward()

  # Grad
  W.data -= step * W.grad

Loss: 3.77138466477363
Loss: 3.1816505370051575
Loss: 2.942427380583093
Loss: 2.8037541346489188
Loss: 2.7458307334736975
Loss: 2.690403383877899
Loss: 2.673328841491543
Loss: 2.6374588219921833
Loss: 2.627834342242151
Loss: 2.6043928171060076
Loss: 2.6054484819936
Loss: 2.58222140913251
Loss: 2.583091144570653
Loss: 2.567479557976471
Loss: 2.574863374821743
Loss: 2.5555404818616547
Loss: 2.5607488186081793
Loss: 2.548132785909326
Loss: 2.558877982027002
Loss: 2.540126746029208
Loss: 2.5472454553188197
Loss: 2.5358368801892373
Loss: 2.548546478537295
Loss: 2.5294014182343245
Loss: 2.537182625011143
Loss: 2.5269594817728476
Loss: 2.5410276123708786
Loss: 2.521400901931985
Loss: 2.529240262791754
Loss: 2.5203706843310907
Loss: 2.535569128173518
Loss: 2.5152898740994245
Loss: 2.522846215950343
Loss: 2.5153743439778404
Loss: 2.531603019499613
Loss: 2.510494425776875
Loss: 2.5175545994788133
Loss: 2.511474522594397
Loss: 2.528665555586343
Loss: 2.5066443426968217
Loss: 2.5130958221232524
Lo

In [68]:
# Negative Log Likelihood
W = torch.randn(27, 27, requires_grad=True, dtype=torch.float64)

runs = 100
step = 100

for _ in range(runs):

  # Forward pass
  logits = W[X] #### This is the ANSWER (directly picks up the row corresponding to index)
  counts = logits.exp()
  probs = counts / counts.sum(dim=1, keepdim=True)
  log_probs = probs.log()
  loss = F.nll_loss(log_probs, Y) + 0.001 * (W**2).mean()
  print(f"Loss: {loss.item()}")

  # Backward pass
  W.grad = None
  loss.backward()

  # Grad
  W.data -= step * W.grad

Loss: 3.8169550748216485
Loss: 3.1379143426777945
Loss: 2.897865726585931
Loss: 2.788709285713348
Loss: 2.7178698349053394
Loss: 2.694563558951431
Loss: 2.6501313939469546
Loss: 2.6367364025880433
Loss: 2.610668481722423
Loss: 2.6105688725343463
Loss: 2.5850086910465615
Loss: 2.585608855720623
Loss: 2.5688671864131796
Loss: 2.5766668873613003
Loss: 2.555233723752911
Loss: 2.5603631610201276
Loss: 2.5469680350338186
Loss: 2.5581964079716855
Loss: 2.5378948354141047
Loss: 2.5447101572489763
Loss: 2.533412423919569
Loss: 2.5466034076583295
Loss: 2.5265161715425313
Loss: 2.53388965100991
Loss: 2.5243162854182195
Loss: 2.538924774754059
Loss: 2.5185352012836946
Loss: 2.525909793595801
Loss: 2.5178948254134443
Loss: 2.533683443271415
Loss: 2.5126625709896406
Loss: 2.5197147190551514
Loss: 2.5131708520458
Loss: 2.530007722236104
Loss: 2.5081807801690648
Loss: 2.5147258734833176
Loss: 2.5095612599627977
Loss: 2.5273350574711357
Loss: 2.5046711332460334
Loss: 2.5106389808956897
Loss: 2.50670156

In [69]:
# Using cross entropy
# Cross Entropy Loss

W = torch.randn(27, 27, requires_grad=True, dtype=torch.float64)

runs = 100
step = 100

for _ in range(runs):

  # Forward pass
  logits = W[X]
  loss = F.cross_entropy(logits, Y) + 0.001 * (W**2).mean()
  print(f"Loss: {loss.item()}")

  # Backward pass
  W.grad = None
  loss.backward()

  # Grad
  W.data -= step * W.grad

Loss: 3.6828773870087064
Loss: 3.105252753427347
Loss: 2.903018896967734
Loss: 2.7886051214862455
Loss: 2.7217638897065313
Loss: 2.680385476996931
Loss: 2.65685570109808
Loss: 2.6478254940095685
Loss: 2.627557366453438
Loss: 2.6248071211515485
Loss: 2.593693622748568
Loss: 2.5900860918024704
Loss: 2.581397476594199
Loss: 2.5884149723610537
Loss: 2.5627172315480555
Loss: 2.5604829847652657
Loss: 2.5569511211652722
Loss: 2.567890325582249
Loss: 2.5445562798861974
Loss: 2.5429224157128716
Loss: 2.5419255588775918
Loss: 2.5548337660832634
Loss: 2.532583285999993
Loss: 2.5313956971788376
Loss: 2.5317520049881654
Loss: 2.5457905498058233
Loss: 2.5241207573151607
Loss: 2.5232746603070977
Loss: 2.52445253734639
Loss: 2.5391860165593636
Loss: 2.517872817226247
Loss: 2.5172756634581677
Loss: 2.5189985805229895
Loss: 2.5341840295430496
Loss: 2.5131078759208774
Loss: 2.5126898578802437
Loss: 2.5147939106134416
Loss: 2.5302877725710577
Loss: 2.509377565476848
Loss: 2.509089444341615
Loss: 2.5114690